# Module 6: Policy Gradient — One-Step Actor-Critic

## Learning Objectives
By completing this notebook, you will:
1. Explain the structural difference between REINFORCE-with-baseline and Actor-Critic
2. Implement one-step Actor-Critic with separate actor and critic networks in PyTorch
3. Understand the TD error as the advantage signal for the actor
4. Compare learning speed and stability against REINFORCE

## Prerequisites
- `01_reinforce_from_scratch.ipynb` (policy networks, baseline concept)
- Temporal-difference learning (Module 3): TD target, bootstrapping

## Estimated Time: 15 minutes

---

## From REINFORCE to Actor-Critic

REINFORCE-with-baseline uses **Monte Carlo returns** — it waits until the episode ends to compute $G_t$. This wastes data and increases variance because the return at early steps depends on all future random decisions.

**Actor-Critic** replaces $G_t$ with a **bootstrapped estimate**:

$$\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$$

This one-step TD error $\delta_t$ is the **advantage** — how much better (or worse) was the action than expected? The actor uses this signal immediately after each step, making updates online rather than at episode end.

| Algorithm | Update frequency | Return estimate | Bias | Variance |
|-----------|-----------------|-----------------|------|----------|
| REINFORCE | Once per episode | Monte Carlo | None | High |
| Actor-Critic | Every step | Bootstrapped (TD) | Some | Low |

In [ ]:
# Purpose: Import dependencies and configure reproducibility
# Key Concept: Same environment and architecture as REINFORCE for a fair comparison

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import gymnasium as gym
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cpu')

env_probe = gym.make('CartPole-v1')
OBS_DIM = env_probe.observation_space.shape[0]  # 4
ACT_DIM = env_probe.action_space.n              # 2
env_probe.close()

print(f"CartPole-v1 | obs_dim={OBS_DIM} | act_dim={ACT_DIM}")

## 1. Actor and Critic Networks

We use **separate** networks for the actor and critic. Sharing parameters can work but requires careful tuning of loss weighting. Separate networks are easier to reason about and debug.

- **Actor**: maps $s_t \to$ action probabilities (same as before)
- **Critic**: maps $s_t \to V(s_t)$, a scalar value estimate

In [ ]:
# Purpose: Define actor and critic network architectures
# Key Concept: Identical hidden size ensures any performance difference is algorithmic, not capacity

class Actor(nn.Module):
    """
    Softmax policy network (actor).

    Parameters
    ----------
    obs_dim : int
        Observation space dimension.
    act_dim : int
        Number of discrete actions.
    hidden_dim : int
        Hidden layer width.
    """

    def __init__(self, obs_dim: int, act_dim: int, hidden_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, act_dim)
        )

    def forward(self, obs: torch.Tensor) -> Categorical:
        return Categorical(logits=self.net(obs))


class Critic(nn.Module):
    """
    Value function network (critic).

    Parameters
    ----------
    obs_dim : int
        Observation space dimension.
    hidden_dim : int
        Hidden layer width.
    """

    def __init__(self, obs_dim: int, hidden_dim: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, obs: torch.Tensor) -> torch.Tensor:
        """Return scalar value estimate V(s)."""
        return self.net(obs).squeeze(-1)


# Verify shapes
actor_test = Actor(OBS_DIM, ACT_DIM)
critic_test = Critic(OBS_DIM)
dummy = torch.zeros(1, OBS_DIM)
print(f"Actor output: {actor_test(dummy).probs}")
print(f"Critic output: {critic_test(dummy).item():.4f}")

## 2. One-Step Actor-Critic Update

At each step $t$, we:
1. Observe $s_t$, select action $a_t \sim \pi_{\theta}(\cdot | s_t)$
2. Execute $a_t$, observe $r_t$, $s_{t+1}$, $\text{done}$
3. Compute TD target: $y_t = r_t + \gamma V(s_{t+1}) \cdot (1 - \text{done})$
4. Compute TD error (advantage): $\delta_t = y_t - V(s_t)$
5. Update critic: minimize $(y_t - V(s_t))^2$
6. Update actor: maximize $\log \pi(a_t | s_t) \cdot \delta_t$

Note: we **detach** the TD target from the computation graph when computing the critic loss, just like in DQN.

In [ ]:
# Purpose: Implement the one-step actor-critic update
# Key Concept: TD error serves as a low-variance advantage signal for the actor

def actor_critic_step(
    obs: np.ndarray,
    next_obs: np.ndarray,
    action: int,
    reward: float,
    done: bool,
    log_prob: torch.Tensor,
    actor: Actor,
    critic: Critic,
    actor_opt: optim.Optimizer,
    critic_opt: optim.Optimizer,
    gamma: float = 0.99
) -> tuple:
    """
    Perform one actor-critic update from a single transition.

    Returns
    -------
    tuple of (actor_loss, critic_loss, td_error)
    """
    obs_t = torch.tensor(obs, dtype=torch.float32)
    next_obs_t = torch.tensor(next_obs, dtype=torch.float32)

    # Step 1: Compute TD target (detach to treat as a fixed label)
    with torch.no_grad():
        next_value = critic(next_obs_t.unsqueeze(0)).squeeze()
        td_target = reward + gamma * next_value * (1.0 - float(done))

    # Step 2: Compute current value estimate
    current_value = critic(obs_t.unsqueeze(0)).squeeze()

    # Step 3: TD error (advantage)
    td_error = td_target - current_value

    # Step 4: Critic loss — minimize squared TD error
    critic_loss = td_error.pow(2)

    critic_opt.zero_grad()
    critic_loss.backward()
    critic_opt.step()

    # Step 5: Actor loss — use detached TD error as advantage
    actor_loss = -log_prob * td_error.detach()

    actor_opt.zero_grad()
    actor_loss.backward()
    actor_opt.step()

    return actor_loss.item(), critic_loss.item(), td_error.item()


print("One-step actor-critic update function defined.")

## 3. Training Loop

Unlike REINFORCE, the actor-critic loop updates at **every step**, not every episode. We log actor loss, critic loss, and episode reward separately to understand what each component learns.

In [ ]:
# Purpose: Full training loop for one-step actor-critic
# Key Concept: Per-step updates mean 500 updates per episode (vs 1 for REINFORCE)

def train_actor_critic(
    n_episodes: int = 400,
    lr_actor: float = 1e-3,
    lr_critic: float = 5e-3,
    gamma: float = 0.99,
    seed: int = SEED
) -> dict:
    """
    Train one-step actor-critic on CartPole-v1.

    Returns
    -------
    dict with keys: 'rewards', 'actor_losses', 'critic_losses', 'smoothed_rewards'
    """
    torch.manual_seed(seed)
    np.random.seed(seed)

    env = gym.make('CartPole-v1')
    actor = Actor(OBS_DIM, ACT_DIM)
    critic = Critic(OBS_DIM)
    actor_opt = optim.Adam(actor.parameters(), lr=lr_actor)
    critic_opt = optim.Adam(critic.parameters(), lr=lr_critic)

    rewards_log = []
    actor_losses_log = []
    critic_losses_log = []

    for ep in range(n_episodes):
        obs, _ = env.reset()
        ep_reward = 0.0
        ep_actor_loss = []
        ep_critic_loss = []
        done = False

        while not done:
            # Select action
            obs_t = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)
            dist = actor(obs_t)
            action = dist.sample()
            log_prob = dist.log_prob(action)

            # Step environment
            next_obs, reward, terminated, truncated, _ = env.step(action.item())
            done = terminated or truncated
            ep_reward += reward

            # Update networks
            a_loss, c_loss, _ = actor_critic_step(
                obs, next_obs, action.item(), reward, done,
                log_prob, actor, critic, actor_opt, critic_opt, gamma
            )
            ep_actor_loss.append(a_loss)
            ep_critic_loss.append(c_loss)
            obs = next_obs

        rewards_log.append(ep_reward)
        actor_losses_log.append(np.mean(ep_actor_loss))
        critic_losses_log.append(np.mean(ep_critic_loss))

    env.close()

    window = 20
    smoothed = np.convolve(rewards_log, np.ones(window) / window, mode='valid')

    return {
        'rewards': rewards_log,
        'actor_losses': actor_losses_log,
        'critic_losses': critic_losses_log,
        'smoothed_rewards': smoothed
    }


print("Training Actor-Critic...")
ac_results = train_actor_critic(n_episodes=400)
print(f"Final 20-ep avg reward: {np.mean(ac_results['rewards'][-20:]):.1f}")

## 4. Baseline REINFORCE for Comparison

We re-run REINFORCE (with baseline) using the same 400 episodes so the comparison is on equal footing. We use the same seed to start from the same random network initialization.

In [ ]:
# Purpose: Train REINFORCE+baseline as a baseline for fair comparison
# Key Concept: Controlling for seed isolates the effect of the algorithm choice

# Re-implement a self-contained REINFORCE+baseline here so this notebook
# can run standalone without importing from notebook 01

class PolicyNet(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(), nn.Linear(hidden, act_dim)
        )
    def forward(self, x):
        return Categorical(logits=self.net(x))


class ValueNet(nn.Module):
    def __init__(self, obs_dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(), nn.Linear(hidden, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)


def train_reinforce_baseline(n_episodes=400, lr=1e-3, gamma=0.99, seed=SEED):
    torch.manual_seed(seed)
    np.random.seed(seed)
    env = gym.make('CartPole-v1')
    policy = PolicyNet(OBS_DIM, ACT_DIM)
    value = ValueNet(OBS_DIM)
    pol_opt = optim.Adam(policy.parameters(), lr=lr)
    val_opt = optim.Adam(value.parameters(), lr=lr)

    rewards_log = []

    for _ in range(n_episodes):
        obs, _ = env.reset()
        log_probs, rewards_ep, obs_list = [], [], []
        done = False
        while not done:
            obs_t = torch.tensor(obs, dtype=torch.float32)
            obs_list.append(obs_t)
            dist = policy(obs_t.unsqueeze(0))
            action = dist.sample()
            log_probs.append(dist.log_prob(action))
            obs, r, term, trunc, _ = env.step(action.item())
            rewards_ep.append(r)
            done = term or trunc

        # Compute normalized returns
        G, returns = 0.0, []
        for r in reversed(rewards_ep):
            G = r + gamma * G
            returns.insert(0, G)
        returns = torch.tensor(returns, dtype=torch.float32)
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)

        obs_stack = torch.stack(obs_list)
        values = value(obs_stack)
        advantages = returns - values.detach()
        log_probs_t = torch.stack(log_probs)

        pol_loss = -(log_probs_t * advantages).sum()
        val_loss = nn.functional.mse_loss(values, returns)

        pol_opt.zero_grad(); pol_loss.backward(); pol_opt.step()
        val_opt.zero_grad(); val_loss.backward(); val_opt.step()

        rewards_log.append(sum(rewards_ep))

    env.close()
    window = 20
    smoothed = np.convolve(rewards_log, np.ones(window) / window, mode='valid')
    return {'rewards': rewards_log, 'smoothed_rewards': smoothed}


print("Training REINFORCE+baseline for comparison...")
reinforce_results = train_reinforce_baseline(n_episodes=400)
print(f"REINFORCE final 20-ep avg: {np.mean(reinforce_results['rewards'][-20:]):.1f}")

## 5. Visualizing Actor Loss, Critic Loss, and Reward Curves

In [ ]:
# Purpose: Visualize all training metrics for actor-critic
# Key Concept: Critic loss should decrease as V(s) converges; actor loss reflects policy improvement

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
window = 20
x_smooth = np.arange(window - 1, 400)

# --- Plot 1: Reward comparison ---
ax = axes[0]
ax.plot(x_smooth, reinforce_results['smoothed_rewards'],
        label='REINFORCE + baseline', color='steelblue', linewidth=2)
ax.plot(x_smooth, ac_results['smoothed_rewards'],
        label='Actor-Critic (1-step)', color='darkorange', linewidth=2)
ax.axhline(y=475, color='green', linestyle='--', alpha=0.7, label='Near-optimal')
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Total Reward (20-ep avg)', fontsize=12)
ax.set_title('Learning Speed Comparison', fontsize=13)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# --- Plot 2: Actor loss over training ---
ax = axes[1]
actor_smooth = np.convolve(ac_results['actor_losses'],
                           np.ones(window) / window, mode='valid')
ax.plot(x_smooth, actor_smooth, color='darkorange', linewidth=2)
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Mean Actor Loss', fontsize=12)
ax.set_title('Actor Loss (policy gradient)', fontsize=13)
ax.grid(True, alpha=0.3)

# --- Plot 3: Critic loss over training ---
ax = axes[2]
critic_smooth = np.convolve(ac_results['critic_losses'],
                            np.ones(window) / window, mode='valid')
ax.plot(x_smooth, critic_smooth, color='crimson', linewidth=2)
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Mean Critic Loss (MSE)', fontsize=12)
ax.set_title('Critic Loss (TD error squared)', fontsize=13)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('actor_critic_training.png', dpi=100, bbox_inches='tight')
plt.show()
print("Figure saved to actor_critic_training.png")

## Exercise 6.3: Shared Network Architecture

**Task:** Instead of separate actor and critic networks, implement a **shared-trunk** architecture where both networks share the first linear layer but have separate heads:

```
obs → Linear(obs_dim, 128) → ReLU → shared_features
    shared_features → Linear(128, act_dim)  [actor head]
    shared_features → Linear(128, 1)        [critic head]
```

**Expected output:** The class should be instantiable and return both an action distribution and a value in a single forward pass.

<details>
<summary>Hint 1</summary>
Store the shared trunk as `self.trunk`, actor head as `self.actor_head`, critic head as `self.critic_head`.
</details>

<details>
<summary>Hint 2 (more specific)</summary>
`forward(self, obs)` should return `(Categorical(logits=actor_head(features)), critic_head(features).squeeze(-1))`.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

class SharedActorCritic(nn.Module):
    """
    Actor-Critic with shared feature extraction trunk.

    Parameters
    ----------
    obs_dim : int
        Observation space dimension.
    act_dim : int
        Number of discrete actions.
    hidden_dim : int
        Width of the shared trunk layer.
    """

    def __init__(self, obs_dim: int, act_dim: int, hidden_dim: int = 128):
        super().__init__()
        # YOUR CODE HERE: define self.trunk, self.actor_head, self.critic_head
        pass

    def forward(self, obs: torch.Tensor):
        """
        Returns
        -------
        tuple of (Categorical distribution, value tensor)
        """
        # YOUR CODE HERE
        pass

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_6_3():
    model = SharedActorCritic(OBS_DIM, ACT_DIM)
    assert model is not None, "SharedActorCritic must be defined"

    dummy = torch.zeros(1, OBS_DIM)
    output = model(dummy)

    assert output is not None, "forward() must not return None"
    assert len(output) == 2, f"Expected tuple of length 2, got {len(output)}"

    dist, value = output
    assert isinstance(dist, Categorical), \
        f"First output must be Categorical distribution, got {type(dist)}"
    assert dist.probs.shape[-1] == ACT_DIM, \
        f"Action probs shape should be ({ACT_DIM},), got {dist.probs.shape}"
    assert value.shape == torch.Size([1]) or value.shape == torch.Size([]), \
        f"Critic output should be scalar, got shape {value.shape}"

    # Check shared trunk exists
    assert hasattr(model, 'trunk'), "Model must have a 'trunk' attribute"
    assert hasattr(model, 'actor_head'), "Model must have an 'actor_head' attribute"
    assert hasattr(model, 'critic_head'), "Model must have a 'critic_head' attribute"

    print("Exercise 6.3 passed! Shared actor-critic architecture correct.")

test_exercise_6_3()

## Exercise 6.4: Critic Learning Rate Sensitivity

**Task:** The critic's learning rate matters: too slow and it can't track the value of a changing policy; too fast and it oscillates. Train actor-critic with three critic learning rates: `lr_critic` in `{1e-4, 5e-3, 1e-2}`, keeping `lr_actor=1e-3` fixed. Run 300 episodes each and report the final 20-episode average reward for each setting.

<details>
<summary>Hint 1</summary>
Use `train_actor_critic(n_episodes=300, lr_actor=1e-3, lr_critic=lr_c, seed=42)` in a loop.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------
critic_lrs = [1e-4, 5e-3, 1e-2]
lr_results = {}  # Map lr_critic -> final_avg_reward

# Fill lr_results with experiments
# Print a summary table at the end

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_6_4():
    assert len(lr_results) == 3, \
        f"Expected 3 lr_critic experiments, got {len(lr_results)}"
    for lr in critic_lrs:
        assert lr in lr_results, f"Missing result for lr_critic={lr}"
        val = lr_results[lr]
        assert isinstance(val, (int, float)), \
            f"lr_results[{lr}] should be a scalar reward, got {type(val)}"
        assert val > 0, f"Final reward must be positive, got {val}"
    print("Exercise 6.4 passed! Critic LR sensitivity analysis complete.")

test_exercise_6_4()

## Key Takeaways

1. **Actor-Critic replaces Monte Carlo returns with bootstrapped TD targets**, updating the policy at every step rather than once per episode. This reduces variance at the cost of some bias.

2. **The TD error $\delta_t = r + \gamma V(s') - V(s)$** is the advantage signal: positive means the action was better than expected, negative means worse. The actor increases/decreases the probability of that action accordingly.

3. **Separate actor and critic networks** allow independent learning rates and are easier to debug. Shared trunks improve sample efficiency but require careful loss weighting.

4. **Critic learning rate matters as much as actor learning rate.** A critic that learns too slowly gives stale advantage estimates; too fast and it overreacts to single transitions.

5. **Actor-Critic is the foundation** for PPO, A3C, SAC, and most modern RL algorithms — the architecture you built here scales to robotics and complex games.

## What's Next

Module 7 addresses the core weakness of standard policy gradient methods: **step size sensitivity**. PPO constrains how large a policy update can be, enabling multiple gradient steps per batch without risk of policy collapse. This is the key to reliable, sample-efficient training at scale.

## Additional Resources

- Sutton & Barto, Section 13.5: Actor-Critic Methods
- Mnih et al. (2016): "Asynchronous Methods for Deep Reinforcement Learning" (A3C)
- Spinning Up: [Actor-Critic](https://spinningup.openai.com/en/latest/algorithms/vpg.html)